# Synthetic Image Benchmark: PyTorch DataLoader vs VVTK + PyTorch DataLoader

1. Generate synthetic PNG images (112×112×3, uint8) into `img/`
2. Store them into a VVTK sharded binary dataset (`./data`)
3. Benchmark **2 epochs** of data loading — no training, just measure dataloader throughput
   - **Run A:** Standard PyTorch Dataset + DataLoader (reads PNGs from disk)
   - **Run B:** VVTKDataset + PyTorch DataLoader (reads from VVTK shards)

In [3]:
import torch
import numpy as np
import os, time
from PIL import Image
from vvtk_dataset import VVTKDataset

## Step 1 — Generate synthetic PNG images

In [11]:
N = 100_000
IMG_DIR = './img'
H, W, C = 112, 112, 3

os.makedirs(IMG_DIR, exist_ok=True)

print(f'Generating {N} synthetic PNG images ({H}x{W}x{C}, uint8) ...')
rng = np.random.default_rng(42)
t0 = time.time()
for i in range(N):
    pixels = rng.integers(0, 256, size=(H, W, C), dtype=np.uint8)
    Image.fromarray(pixels).save(os.path.join(IMG_DIR, f'{i:07d}.png'))
    if (i + 1) % 25_000 == 0:
        print(f'  {i+1}/{N} ({time.time()-t0:.1f}s)')
print(f'Done in {time.time()-t0:.1f}s')

Generating 100000 synthetic PNG images (112x112x3, uint8) ...
  50000/100000 (61.3s)
  100000/100000 (122.7s)
Done in 122.7s


## Step 2 — Store PNGs into VVTK sharded binary dataset

In [12]:
VVTK_PATH = './data/train'
NUM_SHARDS = 32

print(f'Converting {N} PNGs to VVTK shards ...')
t0 = time.time()
with VVTKDataset(VVTK_PATH, mode='wb', num_shards=NUM_SHARDS,
                    compression=['none', 'none']) as ds:
    for i in range(N):
        img = np.array(Image.open(os.path.join(IMG_DIR, f'{i:07d}.png')), dtype=np.uint8)
        img = img.transpose(2, 0, 1)  # HWC -> CHW
        lbl = np.array([i % 1000], dtype=np.int64)  # synthetic label
        ds.add(i, img, lbl)
        if (i + 1) % 25_000 == 0:
            print(f'  {i+1}/{N} ({time.time()-t0:.1f}s)')
print(f'Done in {time.time()-t0:.1f}s')

Converting 100000 PNGs to VVTK shards ...
  25000/100000 (4.9s)
  50000/100000 (9.9s)
  75000/100000 (14.8s)
  100000/100000 (19.8s)
[VVTK] Saved dataset to ./data/train
Done in 19.8s


## Step 3 — Define both Dataset classes

In [13]:
class PNGDataset(torch.utils.data.Dataset):
    """Standard PyTorch Dataset that reads PNGs from disk."""
    def __init__(self, img_dir, n):
        self.img_dir = img_dir
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        img = np.array(Image.open(os.path.join(self.img_dir, f'{idx:07d}.png')), dtype=np.uint8)
        img = torch.from_numpy(img.transpose(2, 0, 1)).float().div(255.0)  # CHW float32
        lbl = torch.tensor(idx % 1000, dtype=torch.long)
        return img, lbl


class VVTKTorchDataset(torch.utils.data.Dataset):
    """PyTorch Dataset backed by VVTKDataset (reads from VVTK shards)."""
    def __init__(self, path):
        self.ds = VVTKDataset(path, mode='rb', compression=['none', 'none'])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img, lbl = self.ds[idx]           # uint8 [3,112,112], int64 [1]
        img = img.float().div(255.0)
        lbl = lbl.squeeze().long()
        return img, lbl

## Step 4 — Benchmark: PyTorch DataLoader with PNG Dataset (2 epochs)

In [14]:
BATCH_SIZE = 128
NUM_WORKERS = 8
NUM_EPOCHS = 2

png_ds = PNGDataset(IMG_DIR, N)
png_loader = torch.utils.data.DataLoader(
    png_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True
)

print(f'PNG DataLoader: {len(png_loader)} batches/epoch, batch_size={BATCH_SIZE}')
print()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    batches = 0
    for imgs, labels in png_loader:
        batches += 1
    elapsed = time.time() - t0
    print(f'  Epoch {epoch}: {elapsed:.2f}s  ({batches} batches, {N/elapsed:.0f} samples/s)')

PNG DataLoader: 782 batches/epoch, batch_size=128

  Epoch 1: 111.35s  (782 batches, 898 samples/s)
  Epoch 2: 5.39s  (782 batches, 18558 samples/s)


## Step 5 — Benchmark: PyTorch DataLoader with VVTK Dataset (2 epochs)

In [15]:
vvtk_ds = VVTKTorchDataset(VVTK_PATH)
vvtk_loader = torch.utils.data.DataLoader(
    vvtk_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True
)

print(f'VVTK + PyTorch DataLoader: {len(vvtk_loader)} batches/epoch, batch_size={BATCH_SIZE}')
print()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    batches = 0
    for imgs, labels in vvtk_loader:
        batches += 1
    elapsed = time.time() - t0
    print(f'  Epoch {epoch}: {elapsed:.2f}s  ({batches} batches, {N/elapsed:.0f} samples/s)')

VVTK + PyTorch DataLoader: 782 batches/epoch, batch_size=128

  Epoch 1: 5.09s  (782 batches, 19652 samples/s)
  Epoch 2: 4.95s  (782 batches, 20187 samples/s)
